In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
file_path = "/content/drive/My Drive/Colab Notebooks"

Mounted at /content/drive


In [ ]:
movies = pd.read_csv(file_path + "/movies.csv", sep=";", header=None)
movies.head()

,0,1
0,1,"(Dinosaur Planet, 2003)"
1,2,"(Isle of Man TT 2004 Review, 2004)"
2,3,"(Character, 1997)"
3,4,"(Paula Abdul's Get Up & Dance, 1994)"
4,5,"(The Rise and Fall of ECW, 2004)"


In [ ]:
ratings = pd.read_csv(file_path + "/customers_rating.csv",sep=";")
ratings.head()

,Cust_Id,Rating,Date,Movie_Id
0,1488844,3.0,2005-09-06,1
1,822109,5.0,2005-05-13,1
2,885013,4.0,2005-10-19,1
3,30878,4.0,2005-12-26,1
4,823519,3.0,2004-05-03,1


In [ ]:
movies.columns = ["id", "raw"]
movies.head()

,id,raw
0,1,"(Dinosaur Planet, 2003)"
1,2,"(Isle of Man TT 2004 Review, 2004)"
2,3,"(Character, 1997)"
3,4,"(Paula Abdul's Get Up & Dance, 1994)"
4,5,"(The Rise and Fall of ECW, 2004)"


In [ ]:
movies[["title", "year"]] = movies["raw"].str.extract(r'\((.*),\s*(\d{4})\)')
movies.head()

,id,raw,title,year
0,1,"(Dinosaur Planet, 2003)",Dinosaur Planet,2003
1,2,"(Isle of Man TT 2004 Review, 2004)",Isle of Man TT 2004 Review,2004
2,3,"(Character, 1997)",Character,1997
3,4,"(Paula Abdul's Get Up & Dance, 1994)",Paula Abdul's Get Up & Dance,1994
4,5,"(The Rise and Fall of ECW, 2004)",The Rise and Fall of ECW,2004


In [ ]:
movies.drop("raw", axis=1, inplace=True)
movies.head()

,id,title,year
0,1,Dinosaur Planet,2003
1,2,Isle of Man TT 2004 Review,2004
2,3,Character,1997
3,4,Paula Abdul's Get Up & Dance,1994
4,5,The Rise and Fall of ECW,2004


In [ ]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4499 entries, 0 to 4498
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      4499 non-null   int64 
 1   title   4499 non-null   object
 2   year    4499 non-null   object
dtypes: int64(1), object(2)
memory usage: 105.6+ KB


In [ ]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24053764 entries, 0 to 24053763
Data columns (total 4 columns):
 #   Column    Dtype  
---  ------    -----  
 0   Cust_Id   int64  
 1   Rating    float64
 2   Date      object 
 3   Movie_Id  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 734.1+ MB


In [ ]:
movies.isnull().sum()

,0
id,0
title,0
year,0


In [ ]:
ratings.isnull().sum()

,0
Cust_Id,0
Rating,0
Date,0
Movie_Id,0


In [ ]:
movies.duplicated().sum()

np.int64(0)

In [ ]:
ratings.duplicated().sum()

np.int64(0)

In [ ]:
ratings['Date'] = pd.to_datetime(ratings['Date'])
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24053764 entries, 0 to 24053763
Data columns (total 4 columns):
 #   Column    Dtype         
---  ------    -----         
 0   Cust_Id   int64         
 1   Rating    float64       
 2   Date      datetime64[ns]
 3   Movie_Id  int64         
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 734.1 MB


In [ ]:
movies.to_parquet(file_path + "/movies_clean.parquet", engine="pyarrow", index=False)

In [ ]:
ratings.to_parquet(file_path + "/ratings_clean.parquet",  engine="pyarrow", index=False)

In [ ]:
import duckdb
import os

In [ ]:
con = duckdb.connect("netflix_db.duckdb")

In [ ]:
movies_path = os.path.join(file_path, "movies_clean.parquet")
ratings_path = os.path.join(file_path, "ratings_clean.parquet")

# Cria views (tabelas virtuais)
con.execute(f"CREATE OR REPLACE VIEW movies AS SELECT * FROM read_parquet('{movies_path}')")
con.execute(f"CREATE OR REPLACE VIEW ratings AS SELECT * FROM read_parquet('{ratings_path}')")

In [ ]:
print(con.execute("SELECT * FROM movies LIMIT 5").df())
print(con.execute("SELECT * FROM ratings LIMIT 5").df())

   id                         title  year
0   1               Dinosaur Planet  2003
1   2    Isle of Man TT 2004 Review  2004
2   3                     Character  1997
3   4  Paula Abdul's Get Up & Dance  1994
4   5      The Rise and Fall of ECW  2004
   Cust_Id  Rating       Date  Movie_Id
0  1488844     3.0 2005-09-06         1
1   822109     5.0 2005-05-13         1
2   885013     4.0 2005-10-19         1
3    30878     4.0 2005-12-26         1
4   823519     3.0 2004-05-03         1


##1. Quantos filmes estão disponíveis no dataset?

In [ ]:
con.execute("SELECT COUNT(DISTINCT id) AS total_filmes FROM movies").df()

,total_filmes
0,4499


##2.Nome dos 5 filmes com melhor média de avaliação

In [ ]:
con.execute("""
    SELECT m.title, AVG(r.rating) AS media
    FROM ratings r
    JOIN movies m ON r.movie_id = m.id
    GROUP BY m.title
    ORDER BY media DESC
    LIMIT 5
""").df()

,title,media
0,Lost: Season 1,4.670989
1,Ghost in the Shell: Stand Alone Complex: 2nd Gig,4.586364
2,The Simpsons: Season 6,4.581296
3,Inu-Yasha,4.554434
4,Lord of the Rings: The Return of the King: Ext...,4.552000


##3. Os 9 anos com menos lançamentos de filmes

In [ ]:
con.execute("""
    SELECT year, COUNT(*) AS qtd
    FROM movies
    GROUP BY year
    ORDER BY qtd ASC
    LIMIT 9
""").df()

,year,qtd
0,1922,1
1,1926,1
2,1917,1
3,1915,1
4,1924,2
5,1931,2
6,1916,2
7,1918,2
8,1929,2


## 4. Quantos filmes têm avaliação ≥ 4.7 na última data de avaliação

In [ ]:
con.execute("""
    WITH ultima_data AS (
        SELECT MAX(date) AS max_date FROM ratings
    )
    SELECT COUNT(DISTINCT movie_id) AS filmes_avaliação_melhor
    FROM ratings, ultima_data
    WHERE rating >= 4.7
      AND date = max_date
""").df()

,filmes_avaliação_melhor
0,780


##5. Os 5 customers que mais avaliaram filmes

In [ ]:
con.execute("""
    SELECT cust_id, COUNT(*) AS total_avaliacoes
    FROM ratings
    GROUP BY cust_id
    ORDER BY total_avaliacoes DESC
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Cust_Id,total_avaliacoes
0,305344,4467
1,387418,4422
2,2439493,4195
3,1664010,4019
4,2118461,3769
